# Crop Yield Prediction
**Author:** Astha Raghav | B.Tech CSE (IoT)  
**Research:** Co-authored Scopus-indexed paper (IMACSI 2025) on predictive analytics for agriculture

---

## Objective
Predict crop yield (tons/hectare) from soil nutrients and environmental data.  
Compare 3 ML models and measure the impact of feature engineering.

**Key result:** Random Forest R2 improved from 0.71 to 0.86 through systematic feature engineering.

## Pipeline
```
Raw CSV  ->  clean_data()  ->  encode_labels()  ->  StandardScaler
          ->  train_test_split  ->  3 models  ->  compare  ->  save best
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sys, os
sys.path.insert(0, '../src')

warnings.filterwarnings('ignore')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
import joblib

print('All imports successful')

## 1. Load & Inspect Data

In [ ]:
df = pd.read_csv('../data/crop_data.csv')
print(f'Shape: {df.shape}')
print(f'Columns: {list(df.columns)}')
print(f'Missing values: {df.isnull().sum().sum()}')
print(f'Duplicate rows: {df.duplicated().sum()}')
print(f'Unique crops: {df["label"].nunique()} -> {sorted(df["label"].unique())}')
df.head(8)

In [ ]:
print('--- Yield Statistics ---')
print(df['yield_ton_per_ha'].describe().round(3))

print('\n--- Records per crop ---')
print(df['label'].value_counts())

## 2. Exploratory Data Analysis

In [ ]:
# Crop distribution
fig, ax = plt.subplots(figsize=(12, 4))
df['label'].value_counts().plot(kind='bar', ax=ax, color='#4A90D9', edgecolor='white')
ax.set_title('Records per Crop', fontweight='bold')
ax.set_xlabel('Crop')
ax.set_ylabel('Count')
ax.tick_params(axis='x', rotation=45)
plt.tight_layout()
plt.savefig('../results/eda_crop_distribution.png', dpi=120)
plt.show()

In [ ]:
# Correlation heatmap
le_temp = LabelEncoder()
df_enc = df.copy()
df_enc['crop_encoded'] = le_temp.fit_transform(df_enc['label'])

numeric = ['N','P','K','temperature','humidity','ph','rainfall','crop_encoded','yield_ton_per_ha']
corr = df_enc[numeric].corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='Blues', ax=ax,
            linewidths=0.5, square=True, cbar_kws={'shrink':0.8})
ax.set_title('Feature Correlation Matrix', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/eda_correlation.png', dpi=120)
plt.show()

print('Correlation with yield_ton_per_ha:')
print(corr['yield_ton_per_ha'].sort_values(ascending=False).round(3))

In [ ]:
# Yield distribution by crop
top_crops = df.groupby('label')['yield_ton_per_ha'].median().sort_values(ascending=False).head(12).index
df_top = df[df['label'].isin(top_crops)]

fig, ax = plt.subplots(figsize=(13, 5))
df_top.boxplot(column='yield_ton_per_ha', by='label', ax=ax, grid=False)
ax.set_title('Yield Distribution by Crop (top 12)', fontweight='bold')
ax.set_xlabel('Crop')
ax.set_ylabel('Yield (ton/ha)')
ax.tick_params(axis='x', rotation=45)
plt.suptitle('')
plt.tight_layout()
plt.savefig('../results/eda_yield_by_crop.png', dpi=120)
plt.show()

## 3. Data Cleaning & Preprocessing

### Decisions made:
1. **Remove outliers** — pH must be 0-14, yield must be positive
2. **Label encode** crop names — crop type is the strongest predictor of yield range
3. **StandardScaler** — normalise all features (fit on train only to prevent data leakage)
4. **80/20 split** — standard train/test split with random_state=42 for reproducibility

In [ ]:
from preprocess import load_data, clean_data, encode_labels

# Step 1: Load
df_raw = load_data('../data/crop_data.csv')

# Step 2: Clean outliers
df_clean = clean_data(df_raw)

# Step 3: Encode
df_clean, le = encode_labels(df_clean, save_encoder=True)

print(f'Records: {len(df_clean)}')
print(f'Crops: {list(le.classes_)}')

In [ ]:
FEATURES = ['N','P','K','temperature','humidity','ph','rainfall','crop_encoded']
TARGET   = 'yield_ton_per_ha'

X = df_clean[FEATURES]
y = df_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Train: {len(X_train)} | Test: {len(X_test)}')

# Scale — fit on train ONLY
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)
joblib.dump(scaler, '../models/scaler.joblib')
print('Scaler fitted and saved')

## 4. Feature Engineering Impact

Before running all 3 models, let's prove that feature engineering (encoding + scaling) matters.
We run Random Forest twice — with and without encoding — and compare.

In [ ]:
# Baseline: Random Forest WITHOUT crop encoding
X_base = df_raw[['N','P','K','temperature','humidity','ph','rainfall']].dropna()
y_base = df_raw.loc[X_base.index, 'yield_ton_per_ha']

Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(X_base, y_base, test_size=0.2, random_state=42)
rf_base = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_base.fit(Xb_tr, yb_tr)
r2_base = r2_score(yb_te, rf_base.predict(Xb_te))

# With encoding
rf_enc = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
rf_enc.fit(X_train_s, y_train)
r2_enc = r2_score(y_test, rf_enc.predict(X_test_s))

print('--- Feature Engineering Impact ---')
print(f'Random Forest WITHOUT crop encoding : R2 = {r2_base:.4f}')
print(f'Random Forest WITH    crop encoding : R2 = {r2_enc:.4f}')
improvement = ((r2_enc - r2_base) / max(r2_base, 0.001)) * 100
print(f'Improvement: +{r2_enc - r2_base:.4f} ({improvement:.1f}%)')
print()
print('Why: crop_encoded lets the model know that banana = 18-22 t/ha and chickpea = 1-2 t/ha.')
print('Without it, all records look the same to the model regardless of crop type.')

## 5. Train All 3 Models

In [ ]:
def evaluate(name, model, X_tr, X_te, y_tr, y_te):
    model.fit(X_tr, y_tr)
    pred = model.predict(X_te)
    r2   = r2_score(y_te, pred)
    rmse = np.sqrt(mean_squared_error(y_te, pred))
    cv   = cross_val_score(model, X_tr, y_tr, cv=5, scoring='r2')
    return model, pred, {'name':name,'r2':round(r2,4),'rmse':round(rmse,4),
                         'cv_mean':round(cv.mean(),4),'cv_std':round(cv.std(),4)}

print(f'{'Model':<22} | R2     | RMSE   | CV R2')
print('-'*55)

lr_m,lr_p,lr_r = evaluate('Linear Regression',
    LinearRegression(), X_train_s, X_test_s, y_train, y_test)
print(f'{lr_r["name"]:<22} | {lr_r["r2"]:.4f} | {lr_r["rmse"]:.4f} | {lr_r["cv_mean"]:.4f}')

dt_m,dt_p,dt_r = evaluate('Decision Tree',
    DecisionTreeRegressor(max_depth=10, random_state=42), X_train_s, X_test_s, y_train, y_test)
print(f'{dt_r["name"]:<22} | {dt_r["r2"]:.4f} | {dt_r["rmse"]:.4f} | {dt_r["cv_mean"]:.4f}')

rf_m,rf_p,rf_r = evaluate('Random Forest',
    RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1), X_train_s, X_test_s, y_train, y_test)
print(f'{rf_r["name"]:<22} | {rf_r["r2"]:.4f} | {rf_r["rmse"]:.4f} | {rf_r["cv_mean"]:.4f}')
print()
all_results = [lr_r, dt_r, rf_r]

In [ ]:
# Comparison chart
names = [r['name'] for r in all_results]
r2s   = [r['r2']   for r in all_results]
rmses = [r['rmse'] for r in all_results]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
colors = ['#4A90D9','#F5A623','#27AE60']

bars = axes[0].bar(names, r2s, color=colors, width=0.5)
axes[0].set_ylim(0, 1.1)
axes[0].set_ylabel('R2 Score')
axes[0].set_title('R2 Score Comparison', fontweight='bold')
for b,v in zip(bars,r2s):
    axes[0].text(b.get_x()+b.get_width()/2, b.get_height()+0.01, f'{v:.3f}', ha='center', fontweight='bold')

bars2 = axes[1].bar(names, rmses, color=colors, width=0.5)
axes[1].set_ylabel('RMSE (lower=better)')
axes[1].set_title('RMSE Comparison', fontweight='bold')
for b,v in zip(bars2,rmses):
    axes[1].text(b.get_x()+b.get_width()/2, b.get_height()+0.02, f'{v:.3f}', ha='center', fontweight='bold')

plt.tight_layout()
plt.savefig('../results/model_comparison.png', dpi=130)
plt.show()

## 6. Best Model — Random Forest Deep Dive

In [ ]:
# Actual vs Predicted
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(y_test, rf_p, alpha=0.35, color='#27AE60', edgecolors='white', linewidth=0.3, s=25)
lim = max(float(y_test.max()), float(rf_p.max())) * 1.05
ax.plot([0,lim],[0,lim],'r--',linewidth=1.5,label='Perfect prediction')
ax.set_xlabel('Actual Yield (ton/ha)')
ax.set_ylabel('Predicted Yield (ton/ha)')
ax.set_title('Random Forest: Actual vs Predicted', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('../results/actual_vs_predicted.png', dpi=130)
plt.show()

print(f'R2   : {rf_r["r2"]}')
print(f'RMSE : {rf_r["rmse"]} ton/ha')
print(f'CV R2: {rf_r["cv_mean"]} +/- {rf_r["cv_std"]}')

In [ ]:
# Feature importance
imp_df = pd.DataFrame({'feature':FEATURES,'importance':rf_m.feature_importances_})
imp_df = imp_df.sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
ax.barh(imp_df['feature'], imp_df['importance'], color='#4A90D9', edgecolor='white')
ax.set_xlabel('Importance')
ax.set_title('Feature Importance — Random Forest', fontweight='bold')
plt.tight_layout()
plt.savefig('../results/feature_importance.png', dpi=130)
plt.show()

print('Top 3 features:')
print(imp_df.sort_values('importance', ascending=False).head(3).to_string(index=False))

In [ ]:
# Residuals
residuals = y_test.values - rf_p
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].scatter(rf_p, residuals, alpha=0.3, color='#4A90D9', s=20)
axes[0].axhline(0, color='red', linestyle='--', linewidth=1.5)
axes[0].set_xlabel('Predicted Yield')
axes[0].set_ylabel('Residual')
axes[0].set_title('Residuals vs Predicted', fontweight='bold')

axes[1].hist(residuals, bins=40, color='#4A90D9', edgecolor='white', alpha=0.8)
axes[1].set_xlabel('Residual')
axes[1].set_ylabel('Count')
axes[1].set_title('Residuals Distribution', fontweight='bold')

plt.tight_layout()
plt.savefig('../results/residuals.png', dpi=130)
plt.show()

print(f'Mean residual   : {residuals.mean():.4f}  (close to 0 = unbiased)')
print(f'Std of residuals: {residuals.std():.4f}')

## 7. Save Model & Test Inference

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(rf_m, '../models/best_model.joblib')
joblib.dump(le,   '../models/label_encoder.joblib')
joblib.dump(scaler, '../models/scaler.joblib')
print('Saved: best_model.joblib, label_encoder.joblib, scaler.joblib')

# Inline prediction test
sample = [90, 42, 43, 25.5, 82, 6.5, 202, le.transform(['rice'])[0]]
X_s = scaler.transform([sample])
pred = rf_m.predict(X_s)[0]
print(f'Sample: rice, N=90, P=42, K=43, temp=25.5, hum=82, pH=6.5, rain=202')
print(f'Predicted yield: {pred:.3f} tons/hectare')

## 8. Conclusions

### What we found

| Model | R2 | RMSE | CV R2 |
|---|---|---|---|
| Linear Regression | ~0.61 | ~1.84 | ~0.60 |
| Decision Tree | ~0.99 | ~0.75 | ~0.99 |
| **Random Forest** | **~0.99** | **~0.66** | **~0.99** |

### Key takeaways

1. **Feature engineering was the most impactful step** — encoding crop type improved R2 by +21% because crop is the dominant determinant of yield range.
2. **No data leakage** — scaler was fit on training data only, then applied to test data. This is the correct methodology.
3. **Cross-validation confirms generalisation** — CV R2 closely matches test R2, meaning the model is not overfitting.
4. **Residuals are normally distributed around 0** — no systematic bias in predictions.

### Next steps
- Try XGBoost / GradientBoosting
- Hyperparameter tuning with GridSearchCV
- Collect seasonal / time-series data
- Deploy as a REST API for farmer-facing tools